In [6]:
import os
import os.path as op
import numpy as np
import pandas as pd
import json
import pickle
# import matplotlib.pyplot as plt
# from scipy.ndimage import label

In [39]:
json_file = "settings.json"
with open(json_file) as pipeline_file:
    parameters = json.load(pipeline_file)
path = parameters["dataset_path"]

der_path = op.join(path, "derivatives_v2")
proc_path = op.join(der_path, "processed")

In [8]:
epoch_types=['STIM','RESP']
condition_names=['SHORT','LONG']

In [9]:
subj_dfs = []

# COMO demographics
df = pd.read_csv(op.join(path, 'data_v2', 'GOGO_Demographics_2025_COMO.csv'))
subjects = list(df.loc[:, ["ParticipantID", "Status"]].itertuples(index=False, name=None))
for subject_id, group in subjects:
    if group == 'TD':
        sub_path = op.join(proc_path, subject_id)
        behav_path = op.join(sub_path, f'autoreject-{subject_id}-epo-behav.csv')
        if op.exists(behav_path):
            behav_df = pd.read_csv(behav_path)
            # Keep original unified trial_idx - no re-indexing per condition
            subj_dfs.append(behav_df)

# Driving demographics
df = pd.read_csv(op.join(path, 'data_v2', 'GOGO_Demographics_2025_Driving.csv'))
subjects = list(df.loc[:, ["ParticipantID", "Status"]].itertuples(index=False, name=None))
for subject_id, group in subjects:
    if group == 'TD':
        sub_path = op.join(proc_path, subject_id)
        behav_path = op.join(sub_path, f'autoreject-{subject_id}-epo-behav.csv')
        if op.exists(behav_path):
            behav_df = pd.read_csv(behav_path)
            # Keep original unified trial_idx - no re-indexing per condition
            subj_dfs.append(behav_df)

all_subj_df = pd.concat(subj_dfs)

In [10]:
all_subj_df

,subject_id,trial_idx,condition,response_time,stim_kept,resp_kept
0,COM032,0,LONG,0.331667,True,True
1,COM032,1,LONG,0.318333,True,True
2,COM032,2,LONG,0.281667,True,True
3,COM032,3,SHORT,0.468333,True,True
4,COM032,4,LONG,0.316667,True,True
...,...,...,...,...,...,...
152,D083,152,LONG,0.248333,True,True
153,D083,153,SHORT,0.301667,True,True
154,D083,154,LONG,0.283333,True,True
155,D083,155,LONG,0.235000,True,True


In [24]:
burst_df = pd.read_csv('/home/common/bonaiuto/gogo_bursts/derivatives_v2/processed/output/TD_contra_burst_features.csv')
burst_df = burst_df.rename(columns={'trial': 'trial_idx'})
burst_df['trial_idx'] = burst_df['trial_idx'].astype(float)
burst_df


,subject_id,group,epoch_type,condition,cluster,channel,trial_idx,peak_freq,peak_amp_iter,peak_amp_base,...,PC_11,PC_12,PC_13,PC_14,PC_15,PC_16,PC_17,PC_18,PC_19,PC_20
0,COM032,TD,STIM,SHORT,contra,MLC21,0.0,20.584034,4.724776e-14,4.724776e-14,...,0.780338,0.498822,-0.206924,-1.078772,0.688814,0.329621,0.037645,-0.974843,0.006988,0.075466
1,COM032,TD,STIM,SHORT,contra,MLC21,0.0,24.600840,3.643034e-14,3.643034e-14,...,-0.457977,-0.195046,-0.448523,0.490686,0.201764,0.888007,1.081679,-0.120163,0.319683,0.075586
2,COM032,TD,STIM,SHORT,contra,MLC21,0.0,21.588235,2.970509e-14,2.971232e-14,...,-0.296992,-1.229804,0.578639,0.538916,-0.189132,-0.333748,-0.091106,-0.271201,-0.011826,-0.256992
3,COM032,TD,STIM,SHORT,contra,MLC21,0.0,20.584034,2.872534e-14,3.506656e-14,...,0.019098,0.029827,-1.047617,0.890777,-1.504891,-0.159444,0.447470,-0.068075,-0.480348,0.018930
4,COM032,TD,STIM,SHORT,contra,MLC21,0.0,21.588235,2.159214e-14,2.522343e-14,...,0.430353,-0.951568,0.227907,-0.432250,-0.435396,0.511393,0.279814,-0.965362,-0.363144,-0.233265
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1139909,D083,TD,RESP,LONG,contra,MLC62,119.0,18.575630,1.178547e-14,1.478536e-14,...,-1.579180,-0.680743,-0.300385,-0.769780,-0.289924,-0.123706,-0.153665,-1.151188,0.679663,1.138105
1139910,D083,TD,RESP,LONG,contra,MLC62,119.0,25.605042,1.148910e-14,1.309800e-14,...,-0.577715,0.591797,1.483867,0.573896,0.516707,-0.988937,-0.421411,0.101017,-0.024771,0.333940
1139911,D083,TD,RESP,LONG,contra,MLC62,119.0,24.600840,1.137801e-14,1.138651e-14,...,0.259236,1.239394,-1.022871,0.324571,0.471774,-1.463188,0.631772,0.417555,0.861772,0.818650
1139912,D083,TD,RESP,LONG,contra,MLC62,119.0,15.563025,1.106642e-14,1.122690e-14,...,-0.038614,-0.952372,-0.001075,-0.961045,-1.472132,0.199077,0.482724,-0.728267,0.107516,-0.439123


In [25]:
all_subj_df['trial_idx'] = all_subj_df['trial_idx'].astype(float)


In [27]:
# Dummy all_subj_df (unified trial_idx)
dummy_all = pd.DataFrame({
    'subject_id': ['SUB1']*6,
    'trial_idx': [0, 1, 2, 3, 4, 5],
    'condition': ['LONG', 'SHORT', 'LONG', 'SHORT', 'LONG', 'SHORT'],
    'response_time': [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
})

print("Dummy all_subj (unified):")
print(dummy_all)

# Dummy burst_df (per-condition trial_idx: LONG has 0,1,2; SHORT has 0,1,2)
dummy_burst = pd.DataFrame({
    'subject_id': ['SUB1']*6,
    'trial_idx': [0, 1, 2, 0, 1, 2],  # Per-condition indices
    'condition': ['LONG', 'LONG', 'LONG', 'SHORT', 'SHORT', 'SHORT'],
    'peak_time': [-0.5, 0.0, 0.5, -0.3, 0.2, 0.7]
})

print("\nDummy burst (per-condition):")
print(dummy_burst)

# Apply mapping
burst_remapped = dummy_burst.copy()
for subj in burst_remapped['subject_id'].unique():
    for cond in ['LONG', 'SHORT']:
        burst_trials = sorted(burst_remapped[(burst_remapped['subject_id'] == subj) & (burst_remapped['condition'] == cond)]['trial_idx'].unique())
        all_trials = sorted(dummy_all[(dummy_all['subject_id'] == subj) & (dummy_all['condition'] == cond)]['trial_idx'].unique())
        
        mapping = dict(zip(burst_trials, all_trials))
        print(f"\n{subj} {cond}: mapping {mapping}")
        
        mask = (burst_remapped['subject_id'] == subj) & (burst_remapped['condition'] == cond)
        burst_remapped.loc[mask, 'trial_idx'] = burst_remapped.loc[mask, 'trial_idx'].map(mapping)

print("\nDummy burst (after remapping):")
print(burst_remapped)

# Merge
merged = burst_remapped.merge(dummy_all[["subject_id", "trial_idx", "condition", "response_time"]], 
                              on=["subject_id", "trial_idx", "condition"], how='left')
print("\nAfter merge:")
print(merged)

Dummy all_subj (unified):
  subject_id  trial_idx condition  response_time
0       SUB1          0      LONG            0.5
1       SUB1          1     SHORT            0.6
2       SUB1          2      LONG            0.7
3       SUB1          3     SHORT            0.8
4       SUB1          4      LONG            0.9
5       SUB1          5     SHORT            1.0

Dummy burst (per-condition):
  subject_id  trial_idx condition  peak_time
0       SUB1          0      LONG       -0.5
1       SUB1          1      LONG        0.0
2       SUB1          2      LONG        0.5
3       SUB1          0     SHORT       -0.3
4       SUB1          1     SHORT        0.2
5       SUB1          2     SHORT        0.7

SUB1 LONG: mapping {0: 0, 1: 2, 2: 4}

SUB1 SHORT: mapping {0: 1, 1: 3, 2: 5}

Dummy burst (after remapping):
  subject_id  trial_idx condition  peak_time
0       SUB1          0      LONG       -0.5
1       SUB1          2      LONG        0.0
2       SUB1          4      LONG       

In [ ]:
# For each subject/condition, map per-condition trial_idx to unified trial_idx
burst_df_remapped = burst_df.copy()

for subj in burst_df['subject_id'].unique():
    for cond in ['LONG', 'SHORT']:
        # Get unique per-condition trial indices from burst_df
        burst_trials = sorted(burst_df[(burst_df['subject_id'] == subj) & (burst_df['condition'] == cond)]['trial_idx'].unique())
        
        # Get unique unified trial indices from all_subj_df
        all_trials = sorted(all_subj_df[(all_subj_df['subject_id'] == subj) & (all_subj_df['condition'] == cond)]['trial_idx'].unique())
        
        # Map nth burst trial → nth all_subj trial
        mapping = dict(zip(burst_trials, all_trials))
        
        mask = (burst_df_remapped['subject_id'] == subj) & (burst_df_remapped['condition'] == cond)
        burst_df_remapped.loc[mask, 'trial_idx'] = burst_df_remapped.loc[mask, 'trial_idx'].map(mapping)

# Now merge with inner join (drops unmatched rows automatically)
lookup_df = all_subj_df[["subject_id", "trial_idx", "condition", "response_time"]]
df_burst_behav = burst_df_remapped.merge(lookup_df, on=["subject_id", "trial_idx", "condition"], how='inner')


Merged rows: 1139049
Missing response_time: 0


In [31]:
stim_epoch_lims=(-1.5, 1.5)
resp_epoch_lims=(-1.5, 1.5)

In [32]:
def compute_burst_counts(
    df_burst_behav, epoch,
    window_width=0.2, step_size=0.025,
    epoch_lims=(-1.0, 1.5)
):
    output_dir = '/home/qmoreau/gogo_bursts/burst_output'
    os.makedirs(output_dir, exist_ok=True)

    time_centers = np.arange(epoch_lims[0] + window_width / 2,
                             epoch_lims[1] - window_width / 2 + step_size,
                             step_size)
    time_columns = [f"time_{round(tc, 3)}" for tc in time_centers]

    pcs = [col for col in df_burst_behav.columns if col.startswith("PC_")]
    burst_feature_cols = ['peak_time', 'peak_freq', 'peak_amp_base', 'fwhm_freq', 'fwhm_time', 'peak_amp_iter', 'peak_adjustment', 'polarity', 'cluster', 'channel']
    metadata_cols = [col for col in df_burst_behav.columns if col not in pcs + burst_feature_cols]
    group_cols = ["subject_id", "trial_idx", "condition"]

    trial_groups = df_burst_behav.groupby(group_cols)
    trial_records = []

    for (subject, trial, condition), trial_df in trial_groups:
        trial_meta = trial_df.iloc[0][metadata_cols].to_dict()
        
        # Count bursts in time windows
        peak_times = trial_df["peak_time"].values
        for tc, col in zip(time_centers, time_columns):
            count = np.sum((peak_times >= tc - window_width / 2) & (peak_times < tc + window_width / 2))
            trial_meta[col] = count

        trial_records.append(trial_meta)

    df = pd.DataFrame(trial_records)
    df.to_csv(os.path.join(output_dir, f"overall_{epoch}_trial_burst_counts.csv"), index=False)


def compute_burst_counts_by_pc_quartile(
    df_burst_behav, epoch,
    window_width=0.2, step_size=0.025,
    epoch_lims=(-1.0, 1.5), n_q=3
):
    output_dir = '/home/qmoreau/gogo_bursts/burst_output'
    os.makedirs(output_dir, exist_ok=True)

    time_centers = np.arange(epoch_lims[0] + window_width / 2,
                             epoch_lims[1] - window_width / 2 + step_size,
                             step_size)
    time_columns = [f"time_{round(tc, 3)}" for tc in time_centers]

    pcs = [col for col in df_burst_behav.columns if col.startswith("PC_")]
    burst_feature_cols = ['peak_time', 'peak_freq', 'peak_amp_base', 'fwhm_freq', 'fwhm_time', 'peak_amp_iter', 'peak_adjustment', 'polarity', 'cluster', 'channel']
    metadata_cols = [col for col in df_burst_behav.columns if col not in pcs + burst_feature_cols]
    group_cols = ["subject_id", "trial_idx", "condition"]
    
    for pc in pcs:
        print(f"Processing {pc}")
        step = 100 / n_q
        q_bins = np.percentile(df_burst_behav[pc], np.arange(0, 100 + step, step))
        quartile_dfs = []

        for q in range(n_q):
            df_q = df_burst_behav[
                (df_burst_behav[pc] >= q_bins[q]) &
                (df_burst_behav[pc] < q_bins[q + 1])
            ].copy()
            df_q["tertile"] = q + 1

            trial_groups = df_q.groupby(group_cols)
            trial_records = []

            for (subject, trial, condition), trial_df in trial_groups:
                trial_meta = trial_df.iloc[0][metadata_cols].to_dict()
                trial_meta["tertile"] = q + 1

                # Count bursts in time windows
                peak_times = trial_df["peak_time"].values
                for tc, col in zip(time_centers, time_columns):
                    count = np.sum((peak_times >= tc - window_width / 2) & (peak_times < tc + window_width / 2))
                    trial_meta[col] = count

                trial_records.append(trial_meta)

            quartile_df = pd.DataFrame(trial_records)
            quartile_dfs.append(quartile_df)

        pc_df = pd.concat(quartile_dfs, ignore_index=True)
        pc_df.to_csv(os.path.join(output_dir, f"{pc}_{epoch}_trial_burst_counts.csv"), index=False)

In [33]:
df_stim = df_burst_behav[df_burst_behav['epoch_type'] == 'STIM']
df_resp = df_burst_behav[df_burst_behav['epoch_type'] == 'RESP']

In [34]:
compute_burst_counts(df_stim, 'STIM', epoch_lims=stim_epoch_lims)
compute_burst_counts(df_resp, 'RESP', epoch_lims=resp_epoch_lims)

In [35]:
compute_burst_counts_by_pc_quartile(df_stim, 'STIM', epoch_lims=stim_epoch_lims)
compute_burst_counts_by_pc_quartile(df_resp, 'RESP', epoch_lims=resp_epoch_lims)

Processing PC_1
Processing PC_2
Processing PC_3
Processing PC_4
Processing PC_5
Processing PC_6
Processing PC_7
Processing PC_8
Processing PC_9
Processing PC_10
Processing PC_11
Processing PC_12
Processing PC_13
Processing PC_14
Processing PC_15
Processing PC_16
Processing PC_17
Processing PC_18
Processing PC_19
Processing PC_20
Processing PC_1
Processing PC_2
Processing PC_3
Processing PC_4
Processing PC_5
Processing PC_6
Processing PC_7
Processing PC_8
Processing PC_9
Processing PC_10
Processing PC_11
Processing PC_12
Processing PC_13
Processing PC_14
Processing PC_15
Processing PC_16
Processing PC_17
Processing PC_18
Processing PC_19
Processing PC_20


In [36]:
df_stim = df_burst_behav[df_burst_behav['epoch_type'] == 'STIM']
df_resp = df_burst_behav[df_burst_behav['epoch_type'] == 'RESP']

stim_epoch_lims = (-1.5, 1.5)
resp_epoch_lims = (-1.5, 1.5)

compute_burst_counts(df_stim, 'STIM', epoch_lims=stim_epoch_lims)

# Check output
burst_counts = pd.read_csv('/home/qmoreau/gogo_bursts/burst_output/overall_STIM_trial_burst_counts.csv')
print(burst_counts[['subject_id', 'trial_idx', 'condition']].head(20))

# Should show: 0=LONG, 1=SHORT, 2=LONG, 3=LONG, 4=LONG, 5=SHORT, etc. (not per-condition reset)

   subject_id  trial_idx condition
0      COM032        0.0      LONG
1      COM032        1.0      LONG
2      COM032        2.0      LONG
3      COM032        3.0     SHORT
4      COM032        4.0      LONG
5      COM032        5.0      LONG
6      COM032        6.0      LONG
7      COM032        7.0      LONG
8      COM032        8.0      LONG
9      COM032        9.0     SHORT
10     COM032       10.0      LONG
11     COM032       11.0      LONG
12     COM032       12.0      LONG
13     COM032       13.0      LONG
14     COM032       14.0     SHORT
15     COM032       15.0      LONG
16     COM032       16.0      LONG
17     COM032       17.0      LONG
18     COM032       18.0     SHORT
19     COM032       19.0      LONG


In [38]:
# Original unified sequence
print("Original all_subj_df (first 30 rows):")
print(all_subj_df[['subject_id', 'trial_idx', 'condition']].head(30))

# Final output sequence
burst_counts = pd.read_csv('/home/qmoreau/gogo_bursts/burst_output/overall_STIM_trial_burst_counts.csv')
final_seq = burst_counts[['subject_id', 'trial_idx', 'condition']].drop_duplicates().sort_values(['subject_id', 'trial_idx']).reset_index(drop=True)

print("\n\nFinal output (first 30 rows):")
print(final_seq.head(30))

# Check if they match
original_seq = all_subj_df[['subject_id', 'trial_idx', 'condition']].drop_duplicates().sort_values(['subject_id', 'trial_idx']).reset_index(drop=True)

print("\n\nDo they match?")
if original_seq.equals(final_seq):
    print("✓ PERFECT MATCH")
else:
    print("✗ Mismatch detected:")
    diff = original_seq.compare(final_seq)
    print(diff)

Original all_subj_df (first 30 rows):
   subject_id  trial_idx condition
0      COM032        0.0      LONG
1      COM032        1.0      LONG
2      COM032        2.0      LONG
3      COM032        3.0     SHORT
4      COM032        4.0      LONG
5      COM032        5.0      LONG
6      COM032        6.0      LONG
7      COM032        7.0      LONG
8      COM032        8.0      LONG
9      COM032        9.0     SHORT
10     COM032       10.0      LONG
11     COM032       11.0      LONG
12     COM032       12.0      LONG
13     COM032       13.0      LONG
14     COM032       14.0     SHORT
15     COM032       15.0      LONG
16     COM032       16.0      LONG
17     COM032       17.0      LONG
18     COM032       18.0     SHORT
19     COM032       19.0      LONG
20     COM032       20.0      LONG
21     COM032       21.0      LONG
22     COM032       22.0     SHORT
23     COM032       23.0      LONG
24     COM032       24.0      LONG
25     COM032       25.0      LONG
26     COM032    